# LNG Freight Routes - Component Analysis

Looking at the underlying components that make up our Freight Routes prices to understand which component is the dominant factor 

For a full explanation of how to import our Freight Routes data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html

### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

This section can remain unchanged for most Spark API users.

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from pprint import pprint
from io import StringIO
import datetime
import time

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.  Instructions for downloading your credentials can be found here:

https://www.sparkcommodities.com/api/introduction.html


In [ ]:
## Input your file location here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

# 2. Defining functions to fetch Routes data

All endpoint information for LNG Freight Routes can be found in on the API website, and in the dedicated code sample:

https://www.sparkcommodities.com/api/lng-freight/routes.html

In [ ]:
## Define function for listing routes reference data

def list_routes_refdata(access_token):

    content = do_api_get_query(uri="/v1.0/routes/", access_token=access_token)

    reldates = content["data"]["sparkReleaseDates"]

    data_dict = content["data"]

    return reldates, data_dict


# Fetch all reference data
reldates, refdata_dict = list_routes_refdata(access_token)

# formatting as a Dataframe
available_df = pd.json_normalize(refdata_dict['routes'])
available_df = available_df.rename(columns={
    'uuid' : 'UUID',
    'via' : 'Via',
    'loadPort.uuid' : 'LoadPortUUID',
    'loadPort.type' : 'LoadPortType',
    'loadPort.region' : 'LoadPortRegion',
    'loadPort.name' : 'LoadPortName',
    'dischargePort.uuid' : 'DischargePortUUID',
    'dischargePort.type' : 'DischargePortType',
    'dischargePort.region' : 'DischargePortRegion',
    'dischargePort.name' : 'DischargePortName',
})

available_df.head(3)

In [ ]:
## Defining function to fetch routes cost data

def fetch_route_data(access_token, route_uuid, release, congestion_laden= None, congestion_ballast= None):

    query_params = "?release-date={}".format(release)
    if congestion_laden is not None:
        query_params += "&congestion-laden-days={}".format(congestion_laden)
    if congestion_ballast is not None:
        query_params += "&congestion-ballast-days={}".format(congestion_ballast)

    uri = "/v1.0/routes/{}/{}".format(route_uuid, query_params)

    content = do_api_get_query(
        uri=uri,
        access_token=access_token,
    )

    data = content["data"]

    return data


## Defining function to fetch historical data for a range of release dates & store as Dataframe

def routes_history(route_uuid, reldates, laden=None, ballast=None):
    
    # initiliasing data dictionary
    my_route  = {
            "ReleaseDate": [],
            "Period": [],
            "CalMonth":[],
            "StartDate": [],
            "EndDate": [],
            "Total": [],
            "Hire": [],
            "Fuel": [],
            "Port": [],
            "Canal": [],
            "Carbon": [],
            "Congestion": [],
        }

    # iterating through release dates
    for r in reldates:
        try:
            my_dict = fetch_route_data(access_token, route_uuid, release=r,  congestion_laden=laden, congestion_ballast=ballast)
        except:
            print('Bad Date: ' + r)
            continue
        
        # Appending all data for full forward curve per release date
        for data in my_dict["dataPoints"]:
            my_route['StartDate'].append(data["deliveryPeriod"]["startAt"])
            my_route['EndDate'].append(data["deliveryPeriod"]["endAt"])
            my_route['Period'].append(data["deliveryPeriod"]["name"])

            my_route['Total'].append(data["costsInUsdPerMmbtu"]["total"])
            my_route['Hire'].append(data["costsInUsdPerMmbtu"]["hire"])
            my_route['Fuel'].append(data["costsInUsdPerMmbtu"]["fuel"])
            my_route['Port'].append(data["costsInUsdPerMmbtu"]["port"])
            my_route['Canal'].append(data["costsInUsdPerMmbtu"]["canal"])
            
            # Add congestion and/or carbon costs if applicable
            try:
                my_route['Congestion'].append(data["costsInUsdPerMmbtu"]["congestion"])
            except:
                 my_route['Congestion'].append(0)
            
            try:
                my_route['Carbon'].append(data["costsInUsdPerMmbtu"]["carbon"])
            except:
                 my_route['Carbon'].append(0)

            my_route['ReleaseDate'].append(r)
            my_route['CalMonth'].append(datetime.datetime.strptime(data["deliveryPeriod"]["startAt"], '%Y-%m-%d').strftime('%b%y'))

        # Sleep function is used to avoid API rate limiting.
        time.sleep(0.2)
    
    # convert into Pandas dataframe
    my_route_df = pd.DataFrame(my_route)

    ## Changing the data types to pandas datetime
    for col in ['ReleaseDate', 'StartDate', 'EndDate']:
        my_route_df[col] = pd.to_datetime(my_route_df[col])

    # sorting data by Release Date (latest first)
    my_route_df = my_route_df.sort_values('ReleaseDate')
    
    ## Changing the data type of these columns from 'string' to numbers.
    for col in ["Total","Hire","Fuel","Port","Canal","Congestion", "Carbon"]:
        my_route_df[col] = pd.to_numeric(my_route_df[col])
    
    return my_route_df

# 3. Defining Inputs

In [ ]:
# Retrieving UUID for route of interest
sab_gat_id = available_df[(available_df["LoadPortName"] == "Sabine Pass") & \
                    (available_df["DischargePortName"] == "Gate")]['UUID'].values[0]

# Example of retrieving a route UUID with a specific via point
#sab_fut_id = available_df[(available_df["LoadPortName"] == "Sabine Pass") & \
#                    (available_df["Via"] == "cogh") & \
#                    (available_df["DischargePortName"] == "Futtsu")]['UUID'].values[0]

print(sab_gat_id)

In [ ]:
# Choosing when to start data from
ix = reldates.index('2026-01-02')

# 4. Calling Data

In [ ]:
# Calling data based on defined functions and inputs above
full_df = routes_history(sab_gat_id, reldates[:ix+1], laden =None, ballast=None)

In [ ]:
# Filtering for a specific month
mdf = full_df[full_df["Period"] == 'Spot (Physical)'].sort_values('ReleaseDate').reset_index(drop=True)
mdf.head(3)

# 5. Plotting

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

sns.set_style('whitegrid')

# Assigning colours for each cost component
comp_colours = {
    'Hire':  '#52BE80',
    'Fuel':  '#008bcc',
    'Port': '#C0392B',
    'Canal':  '#E67E22',
    'Carbon': '#1E8449',
    'Congestion': '#7F8C8D'
}

# Create a copy of mdf to modify for plotting
ms = mdf.copy()
x  = ms['ReleaseDate'].copy()

# Initialising figure
fig1, (ax1, ax12) = plt.subplots(
    2, 1, figsize=(14, 8.6), sharex=True,
    gridspec_kw={'height_ratios': [10, 3], 'hspace': 0.30}
)
fig1.subplots_adjust(top=0.78, bottom=0.10, left=0.06, right=0.96)

# Plotting the components as a cumulative filled curve
cum = np.zeros(len(ms))
for col in comp_colours:
    v   = ms[col].fillna(0).to_numpy(float)
    new = cum + v
    ax1.fill_between(x, cum, new, color=comp_colours[col], alpha=0.75,
                      label=col, linewidth=0)
    cum = new

# Plotting total Routes cost as a line plot
ax1.plot(x, ms['Total'], color='#111111', linewidth=1.5, label='Total Route Cost')

ax1.axhline(0, color='grey', linewidth=0.9)
ax1.set_ylabel('$/MMBtu')
ax1.set_xlim([x.iloc[0] - pd.Timedelta(1, 'day'), x.iloc[-1] + pd.Timedelta(1, 'day')])
ax1.legend(loc='upper left', frameon=True, edgecolor='#dddddd', fontsize=10, ncol=2)


for col in comp_colours:
    ax12.plot(x, ms[col]/ms['Total'], color=comp_colours[col])

ax12.set_ylim(0,1)     
ax12.set_xlabel('Release Date', fontsize=12)
ax12.set_ylabel(f'% Contribution', fontsize=12)
ax12.yaxis.set_major_formatter(PercentFormatter(xmax=1))


plt.show()